# Classes: Triangle as Object

## Introduction

Functions are perfect for avoiding repetitive code (DRY principle). But
when you model real-world things—like a collection of triangles defined
by their three side lengths—a Python class allows you to bundle both the
data and the functions together efficiently. 

In the following code cell, the functions from the previous chapter are
integrated into a class named `TriangleSSS`. 

In [ ]:
class TriangleSSS:
    """Represents a geometric triangle based on its three side lengths.

    Attributes:
        a (float): The length of the first side.
        b (float): The length of the second side.
        c (float): The length of the third side.
    """

    def __init__(self, a: float, b: float, c: float):
        """Initialises the triangle with the three side lengths.

        Args:
            a (float, optional): Length of side a.
            b (float, optional): Length of side b.
            c (float, optional): Length of side c.
        """
        self.a = a
        self.b = b
        self.c = c
        
    def _get_semiperimeter(self):
        """Calculates the semiperimeter of the triangle.

        Returns:
            float: The semiperimeter of the triangle.
        """
        return (self.a + self.b + self.c) / 2
    
    def get_area(self):
        """Calculates the area of the triangle using Heron's formula.

        Returns:
            float: The area of the triangle.
        """
        s = self._get_semiperimeter()
        T = (s * (s - self.a) * (s - self.b) * (s - self.c)) ** (1/2)
        return T
    
    def get_altitude(self, base='a'):
        """Calculates the altitude of the triangle relative to a specific base side.

        Args:
            base (str, optional): The base side to which the altitude relates 
                ('a', 'b', or 'c'). Defaults to 'a'.

        Returns:
            float: The altitude relative to the chosen base side.
        """
        T = self.get_area()    
        if base == 'a':
            h = (2 * T) / self.a
        elif base == 'b':
            h = (2 * T) / self.b
        else:
            h = (2 * T) / self.c
        return h
    
    def get_circumcircle_radius(self):
        """Calculates the radius of the circumcircle of the triangle.

        Returns:
            float: The radius of the circumcircle.
        """
        T = self.get_area()
        n = self.a * self.b * self.c
        d = 4 * T
        r = n / d
        return r
    
    def get_incircle_radius(self):
        """Calculates the radius of the incircle of the triangle.

        Returns:
            float: The radius of the incircle.
        """
        T = self.get_area()
        s = self._get_semiperimeter()
        rho = T / s
        return rho

To use the `TriangleSSS` class, we need to create (instantiate) a new
triangle object. The following code cell shows how to do this. 

In [3]:
triangle = TriangleSSS(3, 4, 5)

This allows us to access the methods of the `TriangleSSS` class. Methods
are simply functions defined inside a class. They are called using *dot
notation*, as shown in the example below. 

In [4]:
triangle.get_area()

6.0

Normally, we call a Python function by passing arguments inside the
parentheses. However, thanks to the `self` argument in our method
definitions, the method already knows the values of `a`, `b`, and `c` (by
accessing `self.a`, `self.b`, and `self.c`). The variable `self` always refers
to the specific instance of our object. 

## Enhanced Triangle Class

In geometry, a triangle is uniquely defined if we know specific
combinations of its side lengths and angles. This is determined by the
four classic congruence criteria: 

**SSS** (Side-Side-Side): All three side lengths are known.

**SAS** (Side-Angle-Side): Two sides and the enclosed angle are known.

**ASA** / **SAA** (Angle-Side-Angle / Side-Angle-Angle): One side and
two angles are known. 

**SsA** (Side-Side-Angle): Two sides and an angle opposite the longer
side are known. 

While our initial `TriangleSSS` class worked perfectly when given three
sides, a truly versatile geometric model should handle any valid
combination. Ideally, we want a single, elegant Python class capable of
dynamically initialising a triangle based on whichever of these four
criteria the user provides. 

The following implementation achieves exactly this by identifying the
given parameters and validating them against mathematical rules before
creating the object. 

In [30]:
class Triangle:
    def __init__(self, a=None, b=None, c=None, alpha=None, beta=None, gamma=None):
        """Initialise a Triangle instance using valid congruence criteria.

        Args:
            a (float, optional): Length of side a. Defaults to None.
            b (float, optional): Length of side b. Defaults to None.
            c (float, optional): Length of side c. Defaults to None.
            alpha (float, optional): Angle alpha opposite to side a. Defaults to None.
            beta (float, optional): Angle beta opposite to side b. Defaults to None.
            gamma (float, optional): Angle gamma opposite to side c. Defaults to None.

        Raises:
            ValueError: If the number of provided parameters is not exactly three,
                if the parameters do not form a unique triangle, or if
                mathematical rules (e.g. triangle inequality) are violated.
        """
        # 1. Initialise all attributes to prevent subsequent AttributeErrors
        self.a = a
        self.b = b
        self.c = c
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

        # 2. Determine which values were actually passed to the constructor
        given_sides = [k for k, v in [('a', a), ('b', b), ('c', c)] if v is not None]
        given_angles = [k for k, v in [('alpha', alpha), ('beta', beta), ('gamma', gamma)] if v is not None]
        
        num_sides = len(given_sides)
        num_angles = len(given_angles)

        # A triangle mathematically always requires exactly 3 parameters
        if num_sides + num_angles != 3:
            raise ValueError("A triangle must be defined by exactly 3 parameters.")

        # 3. Validate mathematical congruence theorems
        if num_sides == 3:
            # --- SSS (Side-Side-Side) ---
            if not (a + b > c and a + c > b and b + c > a):
                raise ValueError("Triangle inequality violated: The sum of two sides must be greater than the third.")

        elif num_sides == 2 and num_angles == 1:
            # --- SAS or SsA (Side-Angle-Side / Side-Side-Angle) ---
            angle_name = given_angles[0]
            
            # Standard mapping: Which angle is opposite which side?
            opposite_mapping = {'alpha': 'a', 'beta': 'b', 'gamma': 'c'}
            opposite_side = opposite_mapping[angle_name]

            if opposite_side in given_sides:
                # Case: SsA (Angle is opposite one of the given sides)
                other_side = [s for s in given_sides if s != opposite_side][0]
                
                val_opposite = {'a': a, 'b': b, 'c': c}[opposite_side]
                val_other = {'a': a, 'b': b, 'c': c}[other_side]
                
                # The angle MUST be opposite the longer side, otherwise it is not uniquely defined
                if val_opposite <= val_other:
                    raise ValueError(
                        f"SsA error: The angle '{angle_name}' must be opposite the longer side. "
                        f"Side '{opposite_side}' ({val_opposite}) must be greater than side '{other_side}' ({val_other})."
                    )
            else:
                # Case: SAS (Angle is enclosed between the two given sides)
                # SAS is always uniquely valid, no further checks required here
                pass

        elif num_sides == 1 and num_angles == 2:
            # --- ASA / SAA (Angle-Side-Angle / Side-Angle-Angle) ---
            # Two angles automatically determine the third. The sum must be less than 180 degrees.
            angle_values = [v for v in [alpha, beta, gamma] if v is not None]
            if sum(angle_values) >= 180:
                raise ValueError("The sum of the given angles must be less than 180 degrees.")

        elif num_angles == 3:
            # --- AAA (Angle-Angle-Angle) ---
            raise ValueError("AAA (Three angles) does not define a unique triangle due to a lack of scaling.")
        
    def __repr__(self):
        data = {
            'a': self.a,
            'b': self.b,
            'c': self.c,
            'alpha': self.alpha,
            'beta': self.beta,
            'gamma': self.gamma,
        }
        return f"{self.__class__.__name__}({data})"

In [31]:
dreieck = Triangle(3, 4, 5)
print(dreieck)

Triangle({'a': 3, 'b': 4, 'c': 5, 'alpha': None, 'beta': None, 'gamma': None})
